In [34]:

import pandas as pd
import numpy as np
import json

# Pancreas

In [29]:
organism = "homo_sapiens"
cell_source = "pancreas"
cell_state = None
reference = "hg19"
paper_doi = "https://doi.org/10.1016/j.cmet.2016.08.020"
table_link = "https://www.ncbi.nlm.nih.gov/pmc/articles/PMC5069352/bin/mmc2.xlsx"

# don't include in header
table_name = "mmc2.xlsx"

header = [
    {
      "organism": organism,
      "cell_source": cell_source,
      "reference": reference,
      "paper_doi": paper_doi,
      "table_link": table_link
    }
]
  

In [12]:
excel = pd.read_excel("paper_degs.xlsx", sheet_name=None, skiprows=4)

In [13]:
excel.keys()

dict_keys(['Overview', 'VariableGenes_Celltypes', 'ExpressedGenes_Celltypes', 'ExpressedGenes_Donors', 'ExpressedGenes_BulkSeq', 'ExpressedGenes_Donors_insilico', 'Cell-type compositions', 'Cell and Mapping statitistics'])

In [14]:
excel["ExpressedGenes_Celltypes"].columns

Index(['Rank', 'Unnamed: 1', 'α-cells', 'β-cells', 'γ-cells', 'δ-cells',
       'ε-cells', 'co-expression', 'unclass endocrine', 'acinar cells',
       'ductal cells', 'MHC class II', 'mast cells', 'PSCs',
       'endothelial cells', 'unclass exocrine'],
      dtype='object')

In [15]:
excel["VariableGenes_Celltypes"].columns

Index(['Rank', 'Unnamed: 1', 'α-cells', 'β-cells', 'γ-cells', 'δ-cells',
       'ε-cells', 'unclass endocrine', 'acinar cells', 'ductal cells',
       'MHC class II', 'mast cells', 'PSCs', 'endothelial cells',
       'Unnamed: 14', 'all cells', 'endocrine cells', 'exocrine cells'],
      dtype='object')

In [16]:
n_top_genes = 50

# VariableGenes_Celltypes: Lists with genes ranked in descending order according to biological variation within the different cell types.
# ExpressedGenes_Celltypes: Lists with genes ranked in descending order according to magnitude of expression for the different cell types (used in Figure 2B). 

# The file is sorted in descending order by most relevant genes (they did not release pvals or logfc)
df = excel["ExpressedGenes_Celltypes"].drop(
    columns=["Rank", "Unnamed: 1", "co-expression"]
    ).map(
        lambda x: x.replace("'", "")
    ).iloc[:n_top_genes].melt(
    ).rename(columns={"variable": "celltype", "value": "gene"})


In [17]:
df.head()

,celltype,gene
0,α-cells,GCG
1,α-cells,TTR
2,α-cells,B2M
3,α-cells,CHGB
4,α-cells,FTL


In [18]:
df.celltype.value_counts()

celltype
α-cells              50
β-cells              50
γ-cells              50
δ-cells              50
ε-cells              50
unclass endocrine    50
acinar cells         50
ductal cells         50
MHC class II         50
mast cells           50
PSCs                 50
endothelial cells    50
unclass exocrine     50
Name: count, dtype: int64

In [19]:
col_pval = "p_val_adj"
col_lfc = "avg_logFC"
col_gene = "gene"
col_ct = "celltype"
col_rank = "pct.1"

In [20]:
min_mean = 10
max_pval = 0.05
min_lfc = 1
max_gene_shares = 4

# filter by criteria
dfc = df # df.query(f"Marker == 1.0 & avg_logFC >= {min_lfc}")

# mask out genes that are shared between max_gene_shares cell type
non_repeat_genes = dfc[col_gene].value_counts()[dfc[col_gene].value_counts() < max_gene_shares].index.values

m = dfc[dfc[col_gene].isin(non_repeat_genes)]

# max number to sample is equal to the min number of genes across all celltype
n_sample = m[col_ct].value_counts().min()

In [21]:
m[col_ct].value_counts()

celltype
endothelial cells    31
acinar cells         30
MHC class II         29
ductal cells         27
unclass exocrine     27
mast cells           22
ε-cells              17
PSCs                 17
α-cells              16
unclass endocrine    16
β-cells              14
δ-cells              11
γ-cells              10
Name: count, dtype: int64

In [22]:
# sample n_sample genes
markers = m.groupby("celltype").head(n_sample)
markers_dict = markers.groupby("celltype")["gene"].apply(lambda x: list(x)).to_dict()

In [23]:
markers.celltype.value_counts()

celltype
α-cells              10
β-cells              10
γ-cells              10
δ-cells              10
ε-cells              10
unclass endocrine    10
acinar cells         10
ductal cells         10
MHC class II         10
mast cells           10
PSCs                 10
endothelial cells    10
unclass exocrine     10
Name: count, dtype: int64

## Find Ensembl ID

In [24]:
import sys
import time

from urllib.parse import urlparse, urlencode
from urllib.request import urlopen, Request
from urllib.error import HTTPError

In [25]:
class EnsemblRestClient(object):
    def __init__(self, server='http://rest.ensembl.org', reqs_per_sec=3):
        self.server = server
        self.reqs_per_sec = reqs_per_sec
        self.req_count = 0
        self.last_req = 0

    def perform_rest_action(self, endpoint, hdrs=None, params=None):
        if hdrs is None:
            hdrs = {}

        if 'Content-Type' not in hdrs:
            hdrs['Content-Type'] = 'application/json'

        if params:
            endpoint += '?' + urlencode(params)

        data = None

        # check if we need to rate limit ourselves
        if self.req_count >= self.reqs_per_sec:
            delta = time.time() - self.last_req
            if delta < 1:
                time.sleep(1 - delta)
            self.last_req = time.time()
            self.req_count = 0
        
        try:
            request = Request(self.server + endpoint, headers=hdrs)
            response = urlopen(request)
            content = response.read()
            if content:
                data = json.loads(content)
            self.req_count += 1

        except HTTPError as e:
            # check if we are being rate limited by the server
            if e.code == 429:
                if 'Retry-After' in e.headers:
                    retry = e.headers['Retry-After']
                    time.sleep(float(retry))
                    self.perform_rest_action(endpoint, hdrs, params)
            else:
                sys.stderr.write('Request failed for {0}: Status code: {1.code} Reason: {1.reason}\n'.format(endpoint, e))
           
        return data

    def get_id(self, species, symbol):
        genes = self.perform_rest_action(
            endpoint='/xrefs/symbol/{0}/{1}'.format(species, symbol), 
            params={'object_type': 'gene'}
        )
        if genes:
            stable_id = genes[0]['id']
            return stable_id
        else:
            return "gene not found"

In [26]:
def run(symbol):
    client = EnsemblRestClient()
    gene_id = client.get_id('human', symbol)
    return gene_id

## Convert to evidence json



In [ ]:
df = markers[[col_ct, col_gene]].rename(columns={col_ct : "cell_type_label", col_gene: "gene"})
df["organism"] = organism
df["cell_source"] = cell_source
df["cell_state"] = cell_state
df["gene_id"] = df["gene"]
df["gene_id"] = df["gene_id"].apply(run)
save = df.to_dict(orient = "records")

In [33]:
save

[{'cell_type_label': 'α-cells',
  'gene': 'GCG',
  'organism': 'homo_sapiens',
  'cell_source': 'pancreas',
  'cell_state': None,
  'gene_id': 'GCG'},
 {'cell_type_label': 'α-cells',
  'gene': 'TM4SF4',
  'organism': 'homo_sapiens',
  'cell_source': 'pancreas',
  'cell_state': None,
  'gene_id': 'TM4SF4'},
 {'cell_type_label': 'α-cells',
  'gene': 'CRYBA2',
  'organism': 'homo_sapiens',
  'cell_source': 'pancreas',
  'cell_state': None,
  'gene_id': 'CRYBA2'},
 {'cell_type_label': 'α-cells',
  'gene': 'CHGA',
  'organism': 'homo_sapiens',
  'cell_source': 'pancreas',
  'cell_state': None,
  'gene_id': 'CHGA'},
 {'cell_type_label': 'α-cells',
  'gene': 'GPX3',
  'organism': 'homo_sapiens',
  'cell_source': 'pancreas',
  'cell_state': None,
  'gene_id': 'GPX3'},
 {'cell_type_label': 'α-cells',
  'gene': 'SPINT2',
  'organism': 'homo_sapiens',
  'cell_source': 'pancreas',
  'cell_state': None,
  'gene_id': 'SPINT2'},
 {'cell_type_label': 'α-cells',
  'gene': 'PEMT',
  'organism': 'homo_sa

## Save evidence

In [ ]:
with open("evidence.json", "w") as f:
    json.dump(save, f, indent = 4) 